# NOAA LCDv2 Station Coverage Investigation

This notebook checks whether stations from `lcdv2-station-list.txt` are good candidates for the launch sites in `Launches.csv`/`Locations.csv`.

Goals:

- parse the LCDv2 station list into station id, latitude, longitude, elevation, and station name
- rebuild the launch-site summary from the raw launch data
- compare every launch facility to its nearest LCDv2 stations
- identify whether the stations already downloaded in `data/*/*.csv` are the nearest practical choices
- flag additional facilities and stations that could expand weather coverage beyond the current joins

Important limitation: `lcdv2-station-list.txt` is a station inventory, not a per-station date coverage inventory. Distance is a necessary screen, but final NOAA pulls still need to confirm that hourly LCD data exists for the launch years.

In [23]:
from pathlib import Path
import math
import re

import numpy as np
import pandas as pd

DATA_DIR = Path("data")
DERIVED_DIR = DATA_DIR / "derived"
STATION_LIST_PATH = Path("lcdv2-station-list.txt")

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)

## Load And Parse NOAA LCDv2 Stations

In [24]:
def load_lcdv2_station_list(path: Path) -> pd.DataFrame:
    rows = []
    with path.open("r", encoding="utf-8", errors="replace") as f:
        for raw in f:
            line = raw.rstrip("\n")
            if not line.strip():
                continue
            # NOAA's station inventory is fixed-width in practice:
            # station id, latitude, longitude, elevation, then station name.
            station_id = line[0:11].strip()
            lat = line[12:20].strip()
            lon = line[21:30].strip()
            elev = line[31:37].strip()
            name = line[41:].strip()
            rows.append(
                {
                    "station_id": station_id,
                    "station_lat": pd.to_numeric(lat, errors="coerce"),
                    "station_lon": pd.to_numeric(lon, errors="coerce"),
                    "station_elevation_m": pd.to_numeric(elev, errors="coerce"),
                    "station_name": name,
                }
            )
    stations = pd.DataFrame(rows)
    stations = stations.dropna(subset=["station_lat", "station_lon"]).drop_duplicates("station_id")
    stations["station_country_prefix"] = stations["station_id"].str[:2]
    return stations

stations = load_lcdv2_station_list(STATION_LIST_PATH)
print(f"Parsed {len(stations):,} LCDv2 stations")
stations.head()

Parsed 24,072 LCDv2 stations


,station_id,station_lat,station_lon,station_elevation_m,station_name,station_country_prefix
0,ACL000BARA9,17.5910,-61.8210,5.0,BARBUDA,AC
1,ACM00078861,17.1167,-61.7833,10.0,COOLIDGE FIELD ANTIGUA (AUX.,AC
2,ACW00011647,17.1333,-61.7833,19.2,ST JOHNS,AC
3,AEI0000OMAA,24.4330,54.6511,26.8,ABU DHABI INTL,AE
4,AEI0000OMAB,23.6167,53.3833,94.0,BUHASA,AE


## Rebuild Launch Facility Summary

In [25]:
launches = pd.read_csv(DATA_DIR / "Launches.csv")
locations = pd.read_csv(DATA_DIR / "Locations.csv")

def infer_facility_group(row) -> str:
    location = str(row.get("Location", ""))
    country_code = str(row.get("Country_Code", "")).upper()
    combined = row.get("Comb Launch Site")
    if country_code != "US":
        return combined

    text = location.lower()
    if "kennedy space center" in text:
        return "Kennedy Space Center"
    if "cape canaveral" in text:
        return "Cape Canaveral Space Force Station"
    if "vandenberg" in text:
        return "Vandenberg Space Force Base"
    if "wallops" in text:
        return "Wallops Flight Facility"
    if "pacific spaceport" in text or "kodiak" in text:
        return "Pacific Spaceport Complex Alaska"
    if "china lake" in text:
        return "China Lake"
    if "edwards" in text:
        return "Edwards Air Force Base"
    if "mojave" in text:
        return "Mojave Air and Space Port"
    if "kauai" in text or "pacific missile range" in text:
        return "Pacific Missile Range Facility"
    return combined

launch_df = launches.copy()
launch_df["launch_time_utc"] = pd.to_datetime(launch_df["Launch Time"], utc=True, errors="coerce")
launch_df["launch_date"] = launch_df["launch_time_utc"].dt.date
launch_df = launch_df.merge(
    locations,
    left_on="Location",
    right_on="Orig_Addr",
    how="left",
    suffixes=("", "_location"),
)
launch_df["Country_Code"] = launch_df["Country_Code"].astype(str).str.upper()
launch_df["facility"] = launch_df.apply(infer_facility_group, axis=1)
launch_df["facility_lat_candidate"] = np.where(
    launch_df["Country_Code"].eq("US"),
    launch_df["Lat"],
    launch_df["Comb Launch Site Lat"],
)
launch_df["facility_lon_candidate"] = np.where(
    launch_df["Country_Code"].eq("US"),
    launch_df["Lon"],
    launch_df["Comb Launch Site Lon"],
)

facility_summary = (
    launch_df.dropna(subset=["facility", "facility_lat_candidate", "facility_lon_candidate"])
    .groupby(["facility", "Country", "Country_Code"], dropna=False)
    .agg(
        launches=("Launch Id", "count"),
        first_launch=("launch_date", "min"),
        last_launch=("launch_date", "max"),
        raw_location_strings=("Location", "nunique"),
        launch_site_labels=("Launch Site", "nunique"),
        facility_lat=("facility_lat_candidate", "mean"),
        facility_lon=("facility_lon_candidate", "mean"),
    )
    .reset_index()
    .rename(columns={"Country_Code": "country_code"})
    .sort_values("launches", ascending=False)
)

# Locations.csv places Pacific Spaceport Complex Alaska near interior Alaska
# (64.200841, -149.493673), which is inconsistent with the Kodiak launch
# site and with the already-downloaded Kodiak/Akhiok LCD stations.
coordinate_overrides = {
    "Pacific Spaceport Complex Alaska": {
        "facility_lat": 57.4350,
        "facility_lon": -152.3390,
        "coordinate_note": "manual override: Kodiak-area PSCA coordinate; Locations.csv appears inland",
    }
}
facility_summary["source_facility_lat"] = facility_summary["facility_lat"]
facility_summary["source_facility_lon"] = facility_summary["facility_lon"]
facility_summary["coordinate_note"] = "Locations.csv"
for facility, override in coordinate_overrides.items():
    mask = facility_summary["facility"] == facility
    for col in ["facility_lat", "facility_lon", "coordinate_note"]:
        facility_summary.loc[mask, col] = override[col]

print(f"{len(facility_summary):,} launch facilities have coordinates")
print(f"{facility_summary['launches'].sum():,} launches have facility coordinates")
facility_summary.head(20)

40 launch facilities have coordinates
6,168 launches have facility coordinates


,facility,Country,country_code,launches,first_launch,last_launch,raw_location_strings,launch_site_labels,facility_lat,facility_lon,source_facility_lat,source_facility_lon,coordinate_note
21,Plesetsk Cosmodrome,Russia,RU,1648,1966-03-17,2021-12-27,11,1,62.926415,40.555239,62.926415,40.555239,Locations.csv
1,Baikonur Cosmodrome,Kazakhstan,KZ,1525,1957-10-04,2021-12-27,21,1,45.964585,63.305243,45.964585,63.305243,Locations.csv
4,Cape Canaveral Space Force Station,United States,US,810,1957-12-06,2021-12-19,22,1,28.502515,-80.566468,28.502515,-80.566468,Locations.csv
34,Vandenberg Space Force Base,United States,US,711,1959-02-28,2021-12-18,16,1,34.683069,-120.590025,34.683069,-120.590025,Locations.csv
8,Guiana SC,French Guiana,GF,311,1970-03-10,2021-12-25,6,1,5.201590,-52.728131,5.201590,-52.728131,Locations.csv
13,Kennedy Space Center,United States,US,192,1967-11-09,2021-12-21,2,1,28.626383,-80.620470,28.626383,-80.620470,Locations.csv
38,Xichang Satellite LC,China,CN,167,1984-01-29,2021-12-29,3,1,27.893551,102.250931,27.893551,102.250931,Locations.csv
11,Jiuquan Satellite LC,China,CN,158,1970-04-24,2021-12-29,6,1,40.984524,100.208695,40.984524,100.208695,Locations.csv
30,Taiyuan Satellite LC,China,CN,99,1988-09-06,2021-12-26,4,1,38.848577,111.607983,38.848577,111.607983,Locations.csv
12,Kapustin Yar,Russia,RU,97,1961-10-27,1999-04-28,3,1,48.575267,45.766175,48.575267,45.766175,Locations.csv


## Current Weather Pulls And Stations Already Used

In [26]:
facility_file_map = {
    "Baikonur Cosmodrome": DATA_DIR / "baikonur_cosmodrome" / "Baikonur_Cosmodrome.csv",
    "Cape Canaveral Space Force Station": DATA_DIR / "cape_canaveral_sfs" / "cape_canaveral_sfs.csv",
    "China Lake": DATA_DIR / "china_lake" / "china_lake.csv",
    "Edwards Air Force Base": DATA_DIR / "edwards_afb" / "edwards_afb.csv",
    "Kennedy Space Center": DATA_DIR / "kennedy_sc" / "kennedy_sc.csv",
    "Mojave Air and Space Port": DATA_DIR / "mojave_air_space_port" / "mojave_air_space_port.csv",
    "Pacific Missile Range Facility": DATA_DIR / "pacific_missile_range" / "pacific_missile_range.csv",
    "Pacific Spaceport Complex Alaska": DATA_DIR / "pacific_spaceport_alaska" / "pacific_spaceport_alaska.csv",
    "Plesetsk Cosmodrome": DATA_DIR / "plesetsk_cosmodrome" / "Plesetsk_Cosmodrome.csv",
    "Vandenberg Space Force Base": DATA_DIR / "vandenberg_sfb" / "vandenberg_sfb.csv",
    "Wallops Flight Facility": DATA_DIR / "wallops_flight_facility" / "wallops_flight_facility.csv",
}

def summarize_weather_file(facility: str, path: Path) -> dict:
    if not path.exists():
        return {"facility": facility, "weather_file": str(path), "file_exists": False}
    df = pd.read_csv(path, usecols=lambda c: c in {"STATION", "DATE"}, low_memory=False)
    dates = pd.to_datetime(df["DATE"], errors="coerce")
    station_ids = sorted(df["STATION"].dropna().astype(str).unique())
    return {
        "facility": facility,
        "weather_file": str(path),
        "file_exists": True,
        "current_station_ids": ", ".join(station_ids),
        "current_station_count": len(station_ids),
        "weather_rows": len(df),
        "weather_start": dates.min(),
        "weather_end": dates.max(),
    }

current_weather = pd.DataFrame(
    [summarize_weather_file(facility, path) for facility, path in facility_file_map.items()]
)

current_weather

,facility,weather_file,file_exists,current_station_ids,current_station_count,weather_rows,weather_start,weather_end
0,Baikonur Cosmodrome,data\baikonur_cosmodrome\Baikonur_Cosmodrome.csv,True,KZM00035953,1,13176,1966-01-01 02:00:00,1991-12-21 02:00:00
1,Cape Canaveral Space Force Station,data\cape_canaveral_sfs\cape_canaveral_sfs.csv,True,"USL000TRDF1, USW00012868",2,517229,1957-12-01 00:00:00,2021-12-31 23:56:00
2,China Lake,data\china_lake\china_lake.csv,True,USW00093104,1,9135,1958-01-01 00:00:00,1958-12-31 23:59:00
3,Edwards Air Force Base,data\edwards_afb\edwards_afb.csv,True,USW00023114,1,41606,1990-01-02 05:10:00,1994-12-30 16:00:00
4,Kennedy Space Center,data\kennedy_sc\kennedy_sc.csv,True,"USW00012886, USW00092821",2,2107798,1978-03-16 19:00:00,2021-12-31 23:59:00
5,Mojave Air and Space Port,data\mojave_air_space_port\mojave_air_space_port.csv,True,USW00003183,1,48009,2020-01-01 00:00:00,2021-12-28 13:20:00
6,Pacific Missile Range Facility,data\pacific_missile_range\pacific_missile_range.csv,True,USW00022501,1,9944,2015-01-01 00:00:00,2015-12-31 23:56:00
7,Pacific Spaceport Complex Alaska,data\pacific_spaceport_alaska\pacific_spaceport_alaska.csv,True,"USI0000PAKH, USW00025501",2,430731,2001-01-01 00:00:00,2021-12-31 23:59:00
8,Plesetsk Cosmodrome,data\plesetsk_cosmodrome\Plesetsk_Cosmodrome.csv,True,RSM00022657,1,57733,1966-01-01 00:00:00,1988-10-08 15:00:00
9,Vandenberg Space Force Base,data\vandenberg_sfb\vandenberg_sfb.csv,True,"USW00093214, USW00093215",2,681548,1959-01-01 00:00:00,2021-12-31 23:58:00


## Distance Utility

In [27]:
def haversine_miles(lat1, lon1, lat2, lon2):
    radius_miles = 3958.7613
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    return 2 * radius_miles * np.arcsin(np.sqrt(a))

def nearest_stations_for_facility(row, top_n=10):
    distances = haversine_miles(
        row["facility_lat"],
        row["facility_lon"],
        stations["station_lat"].to_numpy(),
        stations["station_lon"].to_numpy(),
    )
    nearest = stations.copy()
    nearest["distance_miles"] = distances
    nearest = nearest.nsmallest(top_n, "distance_miles").copy()
    nearest.insert(0, "facility", row["facility"])
    nearest.insert(1, "country_code", row["country_code"])
    nearest.insert(2, "launches", row["launches"])
    nearest.insert(3, "first_launch", row["first_launch"])
    nearest.insert(4, "last_launch", row["last_launch"])
    nearest.insert(5, "facility_lat", row["facility_lat"])
    nearest.insert(6, "facility_lon", row["facility_lon"])
    nearest.insert(7, "coordinate_note", row.get("coordinate_note", "Locations.csv"))
    nearest["distance_rank"] = np.arange(1, len(nearest) + 1)
    return nearest

nearest_all = pd.concat(
    [nearest_stations_for_facility(row, top_n=10) for _, row in facility_summary.iterrows()],
    ignore_index=True,
)

nearest_all.head(20)

,facility,country_code,launches,first_launch,last_launch,facility_lat,facility_lon,coordinate_note,station_id,station_lat,station_lon,station_elevation_m,station_name,station_country_prefix,distance_miles,distance_rank
0,Plesetsk Cosmodrome,RU,1648,1966-03-17,2021-12-27,62.926415,40.555239,Locations.csv,RSM00022657,63.0670,40.3500,108.0,EMCA,RS,11.653629,1
1,Plesetsk Cosmodrome,RU,1648,1966-03-17,2021-12-27,62.926415,40.555239,Locations.csv,RSMU0022648,63.1200,39.2800,35.0,TURCHASOVO USSR,RS,42.147338,2
2,Plesetsk Cosmodrome,RU,1648,1966-03-17,2021-12-27,62.926415,40.555239,Locations.csv,RSMU0022656,63.4700,41.7500,28.0,YEMETSK USSR (EMETSK),RS,52.876649,3
3,Plesetsk Cosmodrome,RU,1648,1966-03-17,2021-12-27,62.926415,40.555239,Locations.csv,RSMU0022749,62.3000,39.5800,100.0,GRYAZNAYA USSR,RS,53.233979,4
4,Plesetsk Cosmodrome,RU,1648,1966-03-17,2021-12-27,62.926415,40.555239,Locations.csv,RSMU0022651,63.7200,40.2300,85.0,KENGOZERO USSR,RS,55.751736,5
5,Plesetsk Cosmodrome,RU,1648,1966-03-17,2021-12-27,62.926415,40.555239,Locations.csv,RSM00022651,63.8000,40.6670,85.0,KHOLMOGORSKAYA,RS,60.458132,6
6,Plesetsk Cosmodrome,RU,1648,1966-03-17,2021-12-27,62.926415,40.555239,Locations.csv,RSMU0022762,62.8700,42.7200,24.0,SEMENOVSKOYE USSR,RS,68.248511,7
7,Plesetsk Cosmodrome,RU,1648,1966-03-17,2021-12-27,62.926415,40.555239,Locations.csv,RSMU0010871,61.6700,40.1800,224.0,NIANDOMA USSR,RS,87.642304,8
8,Plesetsk Cosmodrome,RU,1648,1966-03-17,2021-12-27,62.926415,40.555239,Locations.csv,RSMU0010869,62.1000,42.9000,49.0,SHENKURSK USSR (CHENDOURSK),RS,94.072872,9
9,Plesetsk Cosmodrome,RU,1648,1966-03-17,2021-12-27,62.926415,40.555239,Locations.csv,RSMU0022559,64.2200,41.6700,14.0,KHOLMOGORY USSR,RS,95.722730,10


## Are Current Stations The Nearest Options?

In [28]:
current_station_rows = []
current_weather_ids = current_weather.dropna(subset=["current_station_ids"]).copy()
for _, current in current_weather_ids.iterrows():
    facility = current["facility"]
    facility_row = facility_summary.loc[facility_summary["facility"] == facility]
    if facility_row.empty:
        continue
    facility_row = facility_row.iloc[0]
    ids = [s.strip() for s in str(current["current_station_ids"]).split(",") if s.strip()]
    for station_id in ids:
        station = stations.loc[stations["station_id"] == station_id]
        if station.empty:
            current_station_rows.append(
                {
                    "facility": facility,
                    "current_station_id": station_id,
                    "current_station_found_in_lcdv2_list": False,
                }
            )
            continue
        station = station.iloc[0]
        distance = haversine_miles(
            facility_row["facility_lat"],
            facility_row["facility_lon"],
            station["station_lat"],
            station["station_lon"],
        )
        rank = int(
            (haversine_miles(
                facility_row["facility_lat"],
                facility_row["facility_lon"],
                stations["station_lat"].to_numpy(),
                stations["station_lon"].to_numpy(),
            ) < distance).sum()
            + 1
        )
        current_station_rows.append(
            {
                "facility": facility,
                "launches": facility_row["launches"],
                "country_code": facility_row["country_code"],
                "current_station_id": station_id,
                "current_station_name": station["station_name"],
                "current_station_found_in_lcdv2_list": True,
                "current_station_distance_miles": distance,
                "current_station_distance_rank": rank,
            }
        )

current_station_comparison = pd.DataFrame(current_station_rows).merge(
    current_weather,
    on="facility",
    how="left",
)

current_station_comparison.sort_values(["current_station_distance_rank", "launches"], ascending=[False, False])

,facility,launches,country_code,current_station_id,current_station_name,current_station_found_in_lcdv2_list,current_station_distance_miles,current_station_distance_rank,weather_file,file_exists,current_station_ids,current_station_count,weather_rows,weather_start,weather_end
9,Pacific Spaceport Complex Alaska,7,US,USI0000PAKH,AKHIOK,True,77.267091,3,data\pacific_spaceport_alaska\pacific_spaceport_alaska.csv,True,"USI0000PAKH, USW00025501",2,430731,2001-01-01 00:00:00,2021-12-31 23:59:00
7,Mojave Air and Space Port,3,US,USW00003183,MOJAVE AP,True,0.815349,3,data\mojave_air_space_port\mojave_air_space_port.csv,True,USW00003183,1,48009,2020-01-01 00:00:00,2021-12-28 13:20:00
1,Cape Canaveral Space Force Station,810,US,USL000TRDF1,TRIDENT PIER,True,5.760230,2,data\cape_canaveral_sfs\cape_canaveral_sfs.csv,True,"USL000TRDF1, USW00012868",2,517229,1957-12-01 00:00:00,2021-12-31 23:56:00
12,Vandenberg Space Force Base,711,US,USW00093214,VANDENBERG AFB 72393,True,2.674893,2,data\vandenberg_sfb\vandenberg_sfb.csv,True,"USW00093214, USW00093215",2,681548,1959-01-01 00:00:00,2021-12-31 23:58:00
6,Kennedy Space Center,192,US,USW00092821,TITUSVILLE 7 E,True,4.447363,2,data\kennedy_sc\kennedy_sc.csv,True,"USW00012886, USW00092821",2,2107798,1978-03-16 19:00:00,2021-12-31 23:59:00
10,Pacific Spaceport Complex Alaska,7,US,USW00025501,KODIAK AP 70350,True,22.507354,2,data\pacific_spaceport_alaska\pacific_spaceport_alaska.csv,True,"USI0000PAKH, USW00025501",2,430731,2001-01-01 00:00:00,2021-12-31 23:59:00
4,Edwards Air Force Base,5,US,USW00023114,EDWARDS AFB 72381,True,6.484458,2,data\edwards_afb\edwards_afb.csv,True,USW00023114,1,41606,1990-01-02 05:10:00,1994-12-30 16:00:00
11,Plesetsk Cosmodrome,1648,RU,RSM00022657,EMCA,True,11.653629,1,data\plesetsk_cosmodrome\Plesetsk_Cosmodrome.csv,True,RSM00022657,1,57733,1966-01-01 00:00:00,1988-10-08 15:00:00
0,Baikonur Cosmodrome,1525,KZ,KZM00035953,DZUSALY,True,49.995056,1,data\baikonur_cosmodrome\Baikonur_Cosmodrome.csv,True,KZM00035953,1,13176,1966-01-01 02:00:00,1991-12-21 02:00:00
2,Cape Canaveral Space Force Station,810,US,USW00012868,CAPE KENNEDY AFS 74794,True,1.327739,1,data\cape_canaveral_sfs\cape_canaveral_sfs.csv,True,"USL000TRDF1, USW00012868",2,517229,1957-12-01 00:00:00,2021-12-31 23:56:00


## Three Closest Stations For Every Launch Facility

In [29]:
current_pairs = set()
for _, row in current_station_comparison.dropna(subset=["current_station_id"]).iterrows():
    current_pairs.add((row["facility"], row["current_station_id"]))

three_closest_stations = nearest_all.loc[nearest_all["distance_rank"] <= 3].copy()
three_closest_stations["is_current_downloaded_station"] = [
    (facility, station_id) in current_pairs
    for facility, station_id in zip(
        three_closest_stations["facility"],
        three_closest_stations["station_id"],
    )
]
three_closest_stations["station_country_hint"] = three_closest_stations["station_id"].str[:2]
three_closest_stations = three_closest_stations[
    [
        "facility",
        "country_code",
        "launches",
        "first_launch",
        "last_launch",
        "facility_lat",
        "facility_lon",
        "coordinate_note",
        "distance_rank",
        "station_id",
        "station_name",
        "station_country_hint",
        "station_lat",
        "station_lon",
        "station_elevation_m",
        "distance_miles",
        "is_current_downloaded_station",
    ]
].sort_values(["launches", "facility", "distance_rank"], ascending=[False, True, True])

three_closest_stations

,facility,country_code,launches,first_launch,last_launch,facility_lat,facility_lon,coordinate_note,distance_rank,station_id,station_name,station_country_hint,station_lat,station_lon,station_elevation_m,distance_miles,is_current_downloaded_station
0,Plesetsk Cosmodrome,RU,1648,1966-03-17,2021-12-27,62.926415,40.555239,Locations.csv,1,RSM00022657,EMCA,RS,63.0670,40.3500,108.0,11.653629,True
1,Plesetsk Cosmodrome,RU,1648,1966-03-17,2021-12-27,62.926415,40.555239,Locations.csv,2,RSMU0022648,TURCHASOVO USSR,RS,63.1200,39.2800,35.0,42.147338,False
2,Plesetsk Cosmodrome,RU,1648,1966-03-17,2021-12-27,62.926415,40.555239,Locations.csv,3,RSMU0022656,YEMETSK USSR (EMETSK),RS,63.4700,41.7500,28.0,52.876649,False
10,Baikonur Cosmodrome,KZ,1525,1957-10-04,2021-12-27,45.964585,63.305243,Locations.csv,1,KZM00035953,DZUSALY,KZ,45.5000,64.1000,101.0,49.995056,True
11,Baikonur Cosmodrome,KZ,1525,1957-10-04,2021-12-27,45.964585,63.305243,Locations.csv,2,KZM00035849,KAZALY,KZ,45.7667,62.1167,68.0,58.795241,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
391,Shahrud MTS,IR,1,2020-04-22,2020-04-22,36.406224,55.016269,Locations.csv,2,IRI0000OING,GORGAN,IR,36.9094,54.4013,-7.3,48.688217,False
392,Shahrud MTS,IR,1,2020-04-22,2020-04-22,36.406224,55.016269,Locations.csv,3,IRI0000OINE,KALALEH,IR,37.3833,55.4500,152.4,71.637226,False
380,Tai Rui Barge,NaN,1,2019-06-05,2019-06-05,35.494277,123.796461,Locations.csv,1,KSM00047169,HEUKSANDO,KS,34.6872,125.4510,68.5,108.897648,False
381,Tai Rui Barge,NaN,1,2019-06-05,2019-06-05,35.494277,123.796461,Locations.csv,2,KSA00471963,YEONPYEONGDO,KS,34.6670,125.6830,-999.0,121.015068,False


In [43]:
three_closest_stations[three_closest_stations["distance_rank"] == 1][["facility", "launches", "first_launch", "last_launch", "facility_lat", "facility_lon", "station_id", "station_lat", "station_lon", "station_name", "distance_miles", "is_current_downloaded_station"]].sort_values("distance_miles")

,facility,launches,first_launch,last_launch,facility_lat,facility_lon,station_id,station_lat,station_lon,station_name,distance_miles,is_current_downloaded_station
370,Pacific Missile Range Facility,1,2015-11-04,2015-11-04,22.035267,-159.781957,USW00022501,22.0339,-159.7831,BARKING SANDS 91162,0.119510,True
120,Wallops Flight Facility,49,1960-12-04,2021-08-10,37.935206,-75.466200,USW00093739,37.9372,-75.4661,WALLOPS IS NASA TEST FACILITY 72402,0.137881,True
130,Uchinoura SC,42,1966-09-26,2021-11-09,31.251267,131.076127,JAM00047849,31.2500,131.0830,UCHINOURA,0.415301,False
330,Mojave Air and Space Port,3,2020-05-25,2021-06-30,35.056783,-118.157814,USC00045756,35.0492,-118.1619,MOJAVE,0.572643,False
60,Xichang Satellite LC,167,1984-01-29,2021-12-29,27.893551,102.250931,CHM00056571,27.9000,102.2667,XICHANG,1.060992,False
260,Edwards Air Force Base,5,1990-04-05,1994-08-03,34.992840,-117.883454,USW00053144,34.9883,-117.8647,EDWARDS AFB N AUX AIRFIELD,1.106934,False
30,Vandenberg Space Force Base,711,1959-02-28,2021-12-18,34.683069,-120.590025,USW00093215,34.6667,-120.5833,POINT ARGUELLO AFWB,1.193786,True
20,Cape Canaveral Space Force Station,810,1957-12-06,2021-12-19,28.502515,-80.566468,USW00012868,28.4833,-80.5667,CAPE KENNEDY AFS 74794,1.327739,True
90,Kapustin Yar,97,1961-10-27,1999-04-28,48.575267,45.766175,RSM00034571,48.6000,45.7500,KAPUSTIN JAR,1.861946,False
250,China Lake,6,1958-07-25,1958-08-28,35.655171,-117.657437,USW00093104,35.6864,-117.6908,CHINA LAKE NAF 74612,2.857041,True
